# Experiment 12 — Transfer to LLM buyer

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** The fitted policies come from a random buyer. When the buyer is
also an LLM (natural or mirror), does the same seller policy model still
predict well on the new data? Uses `0423_strategic_buyer/` dataset.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error

def load_buyer_variant(kind):
    '''Load natural or mirror LLM-buyer dataset and preprocess.'''
    dfs = []
    for p in POLICIES:
        try:
            dfs.append(nu.load_llm_buyer_runs(p, kind=kind))
        except FileNotFoundError:
            pass
    if not dfs:
        return None
    raw2 = pd.concat(dfs, ignore_index=True)
    _, turns2 = nu.preprocess_turns(raw2)
    return turns2

def eval_policy_on_test(turns_src, turns_test, policy):
    feat, mdl, alpha = SELECTED[policy]
    meta_tr, X_tr, y_tr, cols = nu.build_design_matrix(turns_src, policy, feat)
    meta_te, X_te, y_te, _    = nu.build_design_matrix(turns_test, policy, feat)
    if len(X_tr) < 30 or len(X_te) < 10:
        return np.nan
    scaler = StandardScaler().fit(X_tr)
    y_tr_logit = nu.logit_clip(y_tr)
    if mdl == 'OLS':
        reg = LinearRegression().fit(scaler.transform(X_tr), y_tr_logit)
    else:
        a = float(alpha) if np.isfinite(alpha) else 1.0
        reg = Ridge(alpha=a).fit(scaler.transform(X_tr), y_tr_logit)
    yhat = nu.sigmoid(reg.predict(scaler.transform(X_te)))
    return float(mean_absolute_error(y_te, yhat))

rows = []
for p in POLICIES:
    # In-sample CV on original buyer
    feat, mdl, alpha = SELECTED[p]
    meta, X, y, _ = nu.build_design_matrix(turns, p, feat)
    pred, _ = nu.grouped_cv_predict(meta, X, y, mdl, n_splits=nu.OUTER_FOLDS,
                                     fixed_alpha=float(alpha) if (mdl == 'Ridge' and np.isfinite(alpha)) else None)
    rows.append({'policy': p, 'setting': 'orig_cv', 'mae': nu.cv_metrics_full(pred)['mae_norm']})

for kind in ['natural', 'mirror']:
    turns_llm = load_buyer_variant(kind)
    if turns_llm is None:
        print(f'LLM-buyer ({kind}) dataset not available; skipping.')
        continue
    for p in POLICIES:
        mae = eval_policy_on_test(turns, turns_llm, p)
        rows.append({'policy': p, 'setting': f'llm_{kind}', 'mae': mae})

transfer = pd.DataFrame(rows).pivot(index='policy', columns='setting', values='mae')
transfer.round(4)